In [1]:
import os
import pickle
import h5py
import numpy as np
import tensorflow as tf

from tensorflow import keras as K


loadPath = '/home/sz4544/EEG-motor-imagery-main/project/'

model_path = os.path.join(loadPath, 'models', 'cwgangp_generator_full_dataset.keras')
scaler_path = os.path.join(loadPath, 'models', 'cwgangp_full_dataset_scaler.pkl')
train_path = os.path.join(loadPath, 'train12720_raw_EEG.h5')

save_path = os.path.join(loadPath, 'cwgangp_synthetic_train_full.h5')

print(model_path)
print(scaler_path)
print(train_path)
print(save_path)

generator = K.models.load_model(model_path, compile=False)

with open(scaler_path, 'rb') as f:
    scaler = pickle.load(f)

f_train = h5py.File(train_path, 'r')
ytr = f_train['tasks'][:]
tr_subjects = f_train['subjects'][:]

ytr = ytr.astype(np.int32)
if np.array_equal(np.unique(ytr), np.array([1, 2, 3, 4])):
    ytr = ytr - 1

print("generator loaded")
print("unique labels:", np.unique(ytr))
print("num train labels:", len(ytr))


def inverse_scale_trials(x_scaled, scaler):
    if x_scaled.ndim == 4:
        x_scaled = np.squeeze(x_scaled, axis=-1)

    n, t, c = x_scaled.shape
    x_flat = x_scaled.reshape(-1, c)
    x_inv = scaler.inverse_transform(x_flat)
    x_inv = x_inv.reshape(n, t, c)
    return x_inv
latent_dim = 128
num_classes = 4
samples_per_class = 1000
all_trials = []
all_labels = []

for cls in range(num_classes):
    print(f"Generating class {cls} ...")
    
    labels = tf.constant(np.full((samples_per_class, 1), cls), dtype=tf.int32)
    noise = tf.random.normal([samples_per_class, latent_dim], dtype=tf.float32)
    
    fake_scaled = generator([noise, labels], training=False).numpy()
    fake_trials = inverse_scale_trials(fake_scaled, scaler).astype(np.float32)
    
    all_trials.append(fake_trials)
    all_labels.append(np.full((samples_per_class,), cls, dtype=np.int32))

synthetic_data = np.concatenate(all_trials, axis=0)
synthetic_labels = np.concatenate(all_labels, axis=0)

print("synthetic_data shape:", synthetic_data.shape)
print("synthetic_labels shape:", synthetic_labels.shape)
print("label counts:", np.bincount(synthetic_labels))
print("min/max:", synthetic_data.min(), synthetic_data.max())

with h5py.File(save_path, 'w') as f:
    f.create_dataset('data', data=synthetic_data)
    f.create_dataset('tasks', data=synthetic_labels)

print("saved synthetic file to:", save_path)


I0000 00:00:1774017416.371752   10856 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


/home/sz4544/EEG-motor-imagery-main/project/models/cwgangp_generator_full_dataset.keras
/home/sz4544/EEG-motor-imagery-main/project/models/cwgangp_full_dataset_scaler.pkl
/home/sz4544/EEG-motor-imagery-main/project/train12720_raw_EEG.h5
/home/sz4544/EEG-motor-imagery-main/project/cwgangp_synthetic_train_full.h5


I0000 00:00:1774017418.407334   10856 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22272 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1774017418.407881   10856 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22253 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9


generator loaded
unique labels: [0 1 2 3]
num train labels: 12720
Generating class 0 ...


I0000 00:00:1774017419.976944   10856 cuda_dnn.cc:461] Loaded cuDNN version 90800
W0000 00:00:1774017422.694346   10856 bfc_allocator.cc:311] Allocator (GPU_0_bfc) ran out of memory trying to allocate 7.34GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
W0000 00:00:1774017423.154625   10856 bfc_allocator.cc:311] Allocator (GPU_0_bfc) ran out of memory trying to allocate 6.73GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
W0000 00:00:1774017423.187426   10856 bfc_allocator.cc:311] Allocator (GPU_0_bfc) ran out of memory trying to allocate 6.73GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


Generating class 1 ...
Generating class 2 ...
Generating class 3 ...
synthetic_data shape: (4000, 640, 64)
synthetic_labels shape: (4000,)
label counts: [1000 1000 1000 1000]
min/max: -0.00015833728 9.32314e-05
saved synthetic file to: /home/sz4544/EEG-motor-imagery-main/project/cwgangp_synthetic_train_full.h5
